## Titanic - PyTorch Model v2

Improved version of `pytorch.ipynb`. Changes vs the first model:
- shuffled mini-batches via `DataLoader` (the original iterated batches in the same fixed order every epoch)
- deeper net with `BatchNorm` + `Dropout` for regularization
- `Adam` + weight decay + LR scheduling instead of plain `SGD`
- early stopping on validation loss (keeps the best weights, not the last epoch's)
- 5-fold stratified cross-validation to get a robust accuracy estimate
- an actual Kaggle `submission.csv` at the end (the original never predicted on the test set)

In [ ]:
import os
from pathlib import Path
import zipfile
import copy

import numpy as np
import pandas as pd
from dotenv import load_dotenv

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)


In [ ]:
load_dotenv(override=True)
api_key = os.getenv('KAGGLE_API_TOKEN')


In [ ]:
path = Path('titanic')
if not path.exists():
    import kaggle  # imported lazily: it authenticates against the Kaggle API on import
    kaggle.api.competition_download_cli(str(path))
    zipfile.ZipFile(f'{path}.zip').extractall(path)


In [ ]:
train_df = pd.read_csv(path/'train.csv')
test_df = pd.read_csv(path/'test.csv')


### Feature engineering

Same engineered features as the first model.

In [ ]:
def add_features(df):
    df['LogFare'] = np.log1p(df['Fare'])
    df['Deck'] = df.Cabin.str[0].map(dict(A="ABC", B="ABC", C="ABC", D="DE", E="DE", F="FG", G="FG"))
    df['Family'] = df.SibSp + df.Parch
    df['Alone'] = df.Family == 0
    df['TicketFreq'] = df.groupby('Ticket')['Ticket'].transform('count')
    df['Title'] = df.Name.str.split(', ', expand=True)[1].str.split('.', expand=True)[0]
    df['Title'] = df.Title.map(dict(Mr="Mr", Miss="Miss", Mrs="Mrs", Master="Master"))

add_features(train_df)
add_features(test_df)


In [ ]:
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Deck',
            'Family', 'Alone', 'TicketFreq', 'Title', 'LogFare']
target = 'Survived'

numeric_features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Family', 'TicketFreq', 'LogFare']
categorical_features = ['Sex', 'Embarked', 'Deck', 'Title']

X = train_df[features]
y = train_df[target].values

print(X.shape, y.shape)


### Preprocessing pipeline

In [ ]:
def make_preprocessor():
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    return ColumnTransformer(transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ])


### Model

A wider/deeper MLP than the original, with `BatchNorm` and `Dropout` for regularization.

In [ ]:
class TitanicNet(nn.Module):
    def __init__(self, input_dim, hidden_dims=(64, 32, 16), dropout=0.3):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


### Training utilities

`DataLoader` shuffles the training set every epoch (fixes a real bug in the original: batches were always iterated in the same order). Training uses `Adam` + weight decay, an LR scheduler that backs off on plateauing validation loss, and early stopping that restores the best-val-loss weights instead of just returning whatever the last epoch happened to produce.

In [ ]:
def make_loaders(X_train, y_train, X_val, y_val, batch_size=32):
    train_ds = TensorDataset(
        torch.tensor(X_train.astype(np.float32)),
        torch.tensor(y_train.astype(np.float32)).unsqueeze(1),
    )
    val_ds = TensorDataset(
        torch.tensor(X_val.astype(np.float32)),
        torch.tensor(y_val.astype(np.float32)).unsqueeze(1),
    )
    # drop_last avoids a batch of size 1, which BatchNorm can't compute statistics for
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader


def train_model(model, train_loader, val_loader, epochs=200, lr=1e-3, weight_decay=1e-4,
                 patience=15, verbose=True):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    best_val_loss = float('inf')
    best_state = None
    epochs_no_improve = 0
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * X_batch.size(0)
        train_loss = running_loss / len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        correct = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                y_pred = model(X_batch)
                loss = criterion(y_pred, y_batch)
                val_loss += loss.item() * X_batch.size(0)
                correct += ((torch.sigmoid(y_pred) > 0.5).float() == y_batch).sum().item()
        val_loss /= len(val_loader.dataset)
        val_acc = correct / len(val_loader.dataset)

        scheduler.step(val_loss)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if verbose and (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | "
                  f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        if epochs_no_improve >= patience:
            if verbose:
                print(f"Early stopping at epoch {epoch+1} (best val loss: {best_val_loss:.4f})")
            break

    model.load_state_dict(best_state)
    return model, history


### Baseline train/val run

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

preprocessor = make_preprocessor()
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
if hasattr(X_train_processed, "toarray"):
    X_train_processed = X_train_processed.toarray()
    X_val_processed = X_val_processed.toarray()

input_dim = X_train_processed.shape[1]
train_loader, val_loader = make_loaders(X_train_processed, y_train, X_val_processed, y_val)

model = TitanicNet(input_dim)
model, history = train_model(model, train_loader, val_loader)


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Train vs Validation Loss')
plt.legend()
plt.show()


### Cross-validation

A single train/val split gives a noisy estimate of how well the model generalizes. 5-fold stratified CV trains the same architecture 5 times on different splits so the reported accuracy is more trustworthy.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_accuracies = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y[train_idx], y[val_idx]

    fold_preprocessor = make_preprocessor()
    X_tr_p = fold_preprocessor.fit_transform(X_tr)
    X_va_p = fold_preprocessor.transform(X_va)
    if hasattr(X_tr_p, "toarray"):
        X_tr_p = X_tr_p.toarray()
        X_va_p = X_va_p.toarray()

    fold_train_loader, fold_val_loader = make_loaders(X_tr_p, y_tr, X_va_p, y_va)
    fold_model = TitanicNet(X_tr_p.shape[1])
    fold_model, fold_history = train_model(fold_model, fold_train_loader, fold_val_loader, verbose=False)

    fold_acc = fold_history['val_acc'][-1]
    # val_acc at early-stopping point corresponds to the restored best-loss epoch's *last computed* acc;
    # recompute directly against the best-loss weights for an accurate number
    fold_model.eval()
    with torch.no_grad():
        val_logits = fold_model(torch.tensor(X_va_p.astype(np.float32)))
        val_preds = (torch.sigmoid(val_logits) > 0.5).float().squeeze()
        fold_acc = (val_preds.numpy() == y_va).mean()

    fold_accuracies.append(fold_acc)
    print(f"Fold {fold+1}: val accuracy = {fold_acc:.4f}")

fold_accuracies = np.array(fold_accuracies)
print(f"\nCV accuracy: {fold_accuracies.mean():.4f} +/- {fold_accuracies.std():.4f}")


### Final model + Kaggle submission

Refit the preprocessor on the full training set, train with a small held-out slice for early stopping, then predict on the Kaggle test set.

In [ ]:
X_final_train, X_final_val, y_final_train, y_final_val = train_test_split(
    X, y, test_size=0.1, random_state=SEED, stratify=y
)

final_preprocessor = make_preprocessor()
X_final_train_p = final_preprocessor.fit_transform(X_final_train)
X_final_val_p = final_preprocessor.transform(X_final_val)
if hasattr(X_final_train_p, "toarray"):
    X_final_train_p = X_final_train_p.toarray()
    X_final_val_p = X_final_val_p.toarray()

final_train_loader, final_val_loader = make_loaders(X_final_train_p, y_final_train, X_final_val_p, y_final_val)
final_model = TitanicNet(X_final_train_p.shape[1])
final_model, final_history = train_model(final_model, final_train_loader, final_val_loader)


In [ ]:
X_test_processed = final_preprocessor.transform(test_df[features])
if hasattr(X_test_processed, "toarray"):
    X_test_processed = X_test_processed.toarray()

final_model.eval()
with torch.no_grad():
    test_logits = final_model(torch.tensor(X_test_processed.astype(np.float32)))
    test_preds = (torch.sigmoid(test_logits) > 0.5).int().squeeze().numpy()

submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': test_preds,
})
submission.to_csv('titanic/submission.csv', index=False)
submission.head()
